# CatAPI Vector RAG Inspection

Учебный Kaggle/Jupyter notebook для полного настоящего RAG-пайплайна проекта `cat-breed-assistant`.

Проверяем реальный путь:

```text
TheCatAPI → pandas preprocessing → chunks → sentence-transformers embeddings → ChromaDB → vector retrieval → RAG prompt → Mistral answer
```

В этом notebook нет lexical fallback и mock answer. Если обязательная библиотека или API key отсутствуют, соответствующая ячейка падает с понятной ошибкой.

## 1. Environment Diagnostics

Смотрим Python/runtime и версии ключевых пакетов, которые Kaggle уже даёт в чистой session.

In [ ]:
import importlib.metadata as metadata
import platform
import sys

print(sys.version)
print(platform.platform())
print('executable:', sys.executable)

packages = [
    'numpy', 'pandas', 'scipy', 'scikit-learn', 'torch', 'transformers',
    'huggingface-hub', 'tokenizers', 'datasets', 'sentence-transformers',
    'chromadb', 'protobuf', 'pydantic', 'mistralai',
]

for package in packages:
    try:
        print(f'{package:24} {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package:24} NOT INSTALLED')

## 2. Kaggle: Clone Repository If Needed

В чистой Kaggle session сначала нужен репозиторий, потому что install-cell читает `requirements-kaggle.txt` из проекта. Если папка уже есть, ячейка ничего не перетирает.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/MataNerdy/cat-breed-assistant.git'
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_PROJECT_DIR = KAGGLE_WORKING / 'cat-breed-assistant'

if KAGGLE_WORKING.exists():
    if KAGGLE_PROJECT_DIR.exists():
        print(f'Repository already exists: {KAGGLE_PROJECT_DIR}')
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(KAGGLE_PROJECT_DIR)], check=True)
        print(f'Cloned repository to {KAGGLE_PROJECT_DIR}')
else:
    print('Not running in Kaggle /kaggle/working environment. Skipping clone cell.')

## 3. Dependency Installation

Запустите эту ячейку один раз в чистой Kaggle session. Она ставит только прямые зависимости notebook из `requirements-kaggle.txt`.

После выполнения ячейки **перезапустите Kaggle Session / Kernel**, затем продолжайте с секции `Post-Restart Validation`. После restart не запускайте install-cell повторно без необходимости.

In [ ]:
from pathlib import Path

requirements_path = Path('/kaggle/working/cat-breed-assistant/requirements-kaggle.txt')
if not requirements_path.exists():
    raise RuntimeError('requirements-kaggle.txt not found. Run the clone cell first.')

%pip install --no-cache-dir -r /kaggle/working/cat-breed-assistant/requirements-kaggle.txt
!python -m pip check

## 4. Post-Restart Validation

После restart kernel/session эта ячейка должна выполниться без ошибок. Здесь намеренно нет `try/except`: если обязательная библиотека не импортируется, notebook должен остановиться.

In [ ]:
from __future__ import annotations

import importlib.metadata as metadata
import json
import math
import os
import re
import subprocess
import sys
from pathlib import Path
from typing import Any

import chromadb
import numpy as np
import pandas as pd
import requests
import datasets
from dotenv import load_dotenv
from mistralai import Mistral
from sentence_transformers import SentenceTransformer

print('Python executable:', sys.executable)
for package in [
    'numpy', 'pandas', 'scipy', 'scikit-learn', 'torch', 'transformers',
    'huggingface-hub', 'tokenizers', 'datasets', 'sentence-transformers',
    'chromadb', 'protobuf', 'pydantic', 'mistralai',
]:
    print(f'{package:24} {metadata.version(package)}')

pip_check = subprocess.run([sys.executable, '-m', 'pip', 'check'], text=True, capture_output=True)
print(pip_check.stdout)
print(pip_check.stderr)
if pip_check.returncode != 0:
    raise RuntimeError('pip check failed. Fix dependency conflicts before running RAG cells.')

## 5. Configuration

Все параметры пайплайна собраны в одной ячейке.

In [ ]:
def find_project_root() -> Path:
    env_root = os.getenv('CAT_BREED_ASSISTANT_ROOT')
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path.cwd().parent, Path('/kaggle/working/cat-breed-assistant'), Path('/kaggle/working')])
    for candidate in candidates:
        root = candidate.resolve()
        if (root / 'app.py').exists() and (root / 'data').exists():
            return root
    raise RuntimeError('Project root not found. Clone the repository first.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CATAPI_BREEDS_URL = 'https://api.thecatapi.com/v1/breeds'
RAW_CATAPI_PATH = PROJECT_ROOT / 'data/raw/catapi_breeds.json'
PROCESSED_DOCS_PATH = PROJECT_ROOT / 'data/processed/catapi_breed_documents.jsonl'
PROCESSED_CHUNKS_PATH = PROJECT_ROOT / 'data/processed/catapi_chunks.jsonl'
CHROMA_PATH = PROJECT_ROOT / 'data/chroma_catapi_notebook'
CHROMA_COLLECTION = 'catapi_breed_chunks_notebook'

EMBEDDING_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
EMBEDDING_BATCH_SIZE = 32
CHUNK_SIZE = 900
CHUNK_OVERLAP = 120
TOP_K = 5
MISTRAL_MODEL = 'mistral-small-latest'

load_dotenv()
CAT_API_KEY = os.getenv('CAT_API_KEY')
MISTRAL_API_KEY = os.getenv('MISTRAL_API_KEY')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CHROMA_PATH:', CHROMA_PATH)
print('EMBEDDING_MODEL_NAME:', EMBEDDING_MODEL_NAME)
print('MISTRAL_MODEL:', MISTRAL_MODEL)
print('CAT_API_KEY configured:', bool(CAT_API_KEY))
print('MISTRAL_API_KEY configured:', bool(MISTRAL_API_KEY))

## 6. Data Loading From TheCatAPI

Загружаем реальные данные из TheCatAPI. Если `CAT_API_KEY` задан в environment/Kaggle Secrets, отправляем его в `x-api-key`. Если ключа нет, пробуем публичный запрос.

In [ ]:
def fetch_catapi_breeds() -> list[dict[str, Any]]:
    headers = {'x-api-key': CAT_API_KEY} if CAT_API_KEY else {}
    response = requests.get(CATAPI_BREEDS_URL, headers=headers, timeout=30)
    response.raise_for_status()
    data = response.json()
    if not isinstance(data, list):
        raise ValueError('TheCatAPI response must be a list of breed records.')
    return data

raw_breeds = fetch_catapi_breeds()
RAW_CATAPI_PATH.parent.mkdir(parents=True, exist_ok=True)
RAW_CATAPI_PATH.write_text(json.dumps(raw_breeds, ensure_ascii=False, indent=2), encoding='utf-8')

raw_df = pd.DataFrame(raw_breeds)
print('Raw breed records:', len(raw_df))
print('Raw fields:', list(raw_df.columns))
display(raw_df.head(5))
print('Missing values per selected field:')
display(raw_df[['id', 'name', 'description', 'temperament', 'origin', 'life_span']].isna().sum())
print('Unique breeds:', raw_df['name'].nunique())

## 7. Preprocessing

Показываем выбор полей, нормализацию текста, создание metadata и удаление дубликатов.

In [ ]:
USEFUL_COLUMNS = [
    'id', 'name', 'temperament', 'origin', 'description', 'life_span', 'weight',
    'wikipedia_url', 'reference_image_id', 'grooming', 'energy_level',
    'health_issues', 'child_friendly', 'dog_friendly', 'stranger_friendly',
    'intelligence', 'hairless', 'shedding_level', 'social_needs',
    'vocalisation', 'hypoallergenic',
]
existing_columns = [column for column in USEFUL_COLUMNS if column in raw_df.columns]
selected_df = raw_df[existing_columns].copy()
print('Before preprocessing:')
display(selected_df.head(5))
print('Shape:', selected_df.shape)

In [ ]:
def normalize_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    if isinstance(value, dict):
        return ', '.join(f'{key}: {item}' for key, item in value.items() if item not in (None, ''))
    return re.sub(r'\s+', ' ', str(value)).strip()

def yes_no(value: Any) -> str:
    if value == 1:
        return 'yes'
    if value == 0:
        return 'no'
    return normalize_text(value) or 'unknown'

def build_document_text(row: pd.Series) -> str:
    lines = [
        f"Breed name: {normalize_text(row.get('name'))}",
        f"Description: {normalize_text(row.get('description'))}",
        f"Temperament: {normalize_text(row.get('temperament'))}",
        f"Origin: {normalize_text(row.get('origin'))}",
        f"Life span: {normalize_text(row.get('life_span'))} years",
        f"Weight: {normalize_text(row.get('weight'))}",
        f"Grooming level: {normalize_text(row.get('grooming'))}",
        f"Energy level: {normalize_text(row.get('energy_level'))}",
        f"Health issues score: {normalize_text(row.get('health_issues'))}",
        f"Child friendly score: {normalize_text(row.get('child_friendly'))}",
        f"Dog friendly score: {normalize_text(row.get('dog_friendly'))}",
        f"Stranger friendly score: {normalize_text(row.get('stranger_friendly'))}",
        f"Intelligence score: {normalize_text(row.get('intelligence'))}",
        f"Hairless: {yes_no(row.get('hairless'))}",
        f"Shedding level: {normalize_text(row.get('shedding_level'))}",
        f"Social needs score: {normalize_text(row.get('social_needs'))}",
        f"Vocalisation score: {normalize_text(row.get('vocalisation'))}",
        f"Hypoallergenic: {yes_no(row.get('hypoallergenic'))}",
        f"Wikipedia URL: {normalize_text(row.get('wikipedia_url'))}",
    ]
    return '\n'.join(line for line in lines if not line.endswith(': '))

def build_metadata(row: pd.Series) -> dict[str, str | int | float | bool]:
    metadata = {
        'breed_id': normalize_text(row.get('id')),
        'breed_name': normalize_text(row.get('name')),
        'source': 'thecatapi',
        'origin': normalize_text(row.get('origin')),
        'temperament': normalize_text(row.get('temperament')),
        'life_span': normalize_text(row.get('life_span')),
        'wikipedia_url': normalize_text(row.get('wikipedia_url')),
        'reference_image_id': normalize_text(row.get('reference_image_id')),
        'source_url': CATAPI_BREEDS_URL,
    }
    return {key: value for key, value in metadata.items() if value != ''}

processed_df = selected_df.copy()
processed_df['breed_id'] = processed_df['id'].map(normalize_text)
processed_df['breed_name'] = processed_df['name'].map(normalize_text)
processed_df['document_text'] = processed_df.apply(build_document_text, axis=1)
processed_df['metadata'] = processed_df.apply(build_metadata, axis=1)
processed_df = processed_df[processed_df['document_text'].str.len() > 0].copy()
processed_df = processed_df.drop_duplicates(subset=['breed_id']).reset_index(drop=True)

print('After preprocessing:')
display(processed_df[['breed_id', 'breed_name', 'origin', 'document_text', 'metadata']].head(5))
print('Shape:', processed_df.shape)
print('Unique breeds:', processed_df['breed_name'].nunique())

In [ ]:
documents: list[dict[str, Any]] = []
for row in processed_df.itertuples(index=False):
    documents.append({
        'doc_id': f"catapi_{getattr(row, 'breed_id')}",
        'breed_id': getattr(row, 'breed_id'),
        'breed_name': getattr(row, 'breed_name'),
        'source': 'thecatapi',
        'text': getattr(row, 'document_text'),
        'metadata': getattr(row, 'metadata'),
    })

PROCESSED_DOCS_PATH.parent.mkdir(parents=True, exist_ok=True)
with PROCESSED_DOCS_PATH.open('w', encoding='utf-8') as output_file:
    for document in documents:
        output_file.write(json.dumps(document, ensure_ascii=False) + '\n')

print('Documents:', len(documents))
print('Saved to:', PROCESSED_DOCS_PATH)
print('Document fields:', sorted(documents[0].keys()))
print('Metadata fields:', sorted(documents[0]['metadata'].keys()))

## 8. Chunking

Реализуем chunking прямо в notebook. Chunk сохраняет связь с исходным документом и metadata.

In [ ]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    if chunk_size <= 0:
        raise ValueError('chunk_size must be positive.')
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError('overlap must be >= 0 and smaller than chunk_size.')
    clean_text = normalize_text(text)
    if not clean_text:
        return []
    text_chunks = []
    start = 0
    while start < len(clean_text):
        end = min(start + chunk_size, len(clean_text))
        chunk = clean_text[start:end].strip()
        if chunk:
            text_chunks.append(chunk)
        if end == len(clean_text):
            break
        start = end - overlap
    return text_chunks

chunks: list[dict[str, Any]] = []
for document in documents:
    for chunk_index, chunk in enumerate(chunk_text(document['text'], CHUNK_SIZE, CHUNK_OVERLAP)):
        chunk_id = f"{document['doc_id']}_chunk_{chunk_index:03d}"
        chunks.append({
            'chunk_id': chunk_id,
            'doc_id': document['doc_id'],
            'breed_id': document['breed_id'],
            'breed_name': document['breed_name'],
            'source': document['source'],
            'text': chunk,
            'metadata': {**document['metadata'], 'chunk_id': chunk_id, 'doc_id': document['doc_id'], 'chunk_index': chunk_index},
        })

with PROCESSED_CHUNKS_PATH.open('w', encoding='utf-8') as output_file:
    for chunk in chunks:
        output_file.write(json.dumps(chunk, ensure_ascii=False) + '\n')

chunk_lengths = [len(chunk['text']) for chunk in chunks]
print('Source documents:', len(documents))
print('Chunks:', len(chunks))
print('Average chunk length:', round(float(np.mean(chunk_lengths)), 2))
print('Min chunk length:', min(chunk_lengths))
print('Max chunk length:', max(chunk_lengths))
display(pd.DataFrame(chunks)[['chunk_id', 'breed_name', 'text', 'metadata']].head(5))

## 9. Embeddings

Используем реальную multilingual модель `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`. Модель скачивается с Hugging Face Hub через `sentence-transformers`.

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
chunk_texts = [chunk['text'] for chunk in chunks]
embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=EMBEDDING_BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print('Embedding model:', EMBEDDING_MODEL_NAME)
print('Batch size:', EMBEDDING_BATCH_SIZE)
print('Embedding dimension:', embeddings.shape[1])
print('Embedding matrix shape:', embeddings.shape)
print('First embedding preview:', embeddings[0][:10])
print('Contains NaN:', bool(np.isnan(embeddings).any()))
assert len(embeddings) == len(chunks)
assert embeddings.ndim == 2
assert not np.isnan(embeddings).any()

## 10. ChromaDB Index

Создаём persistent ChromaDB collection. Перед повторной индексацией удаляем старую тестовую collection, чтобы не копить дубликаты.

In [ ]:
def clean_metadata(metadata: dict[str, Any]) -> dict[str, str | int | float | bool]:
    clean: dict[str, str | int | float | bool] = {}
    for key, value in metadata.items():
        if value is None:
            continue
        if isinstance(value, (str, int, float, bool)):
            clean[key] = value
        else:
            clean[key] = str(value)
    return clean

client = chromadb.PersistentClient(path=str(CHROMA_PATH))
existing_collection_names = [collection.name for collection in client.list_collections()]
if CHROMA_COLLECTION in existing_collection_names:
    client.delete_collection(CHROMA_COLLECTION)

collection = client.create_collection(name=CHROMA_COLLECTION, metadata={'hnsw:space': 'cosine'})
ids = [chunk['chunk_id'] for chunk in chunks]
metadatas = [clean_metadata(chunk['metadata']) for chunk in chunks]

batch_size = 64
for start in range(0, len(chunks), batch_size):
    end = start + batch_size
    collection.add(
        ids=ids[start:end],
        documents=chunk_texts[start:end],
        metadatas=metadatas[start:end],
        embeddings=embeddings[start:end].tolist(),
    )

print('Collection:', CHROMA_COLLECTION)
print('Collection count:', collection.count())
peek = collection.peek(3)
print('Sample ids:', peek.get('ids'))
print('Sample metadata:')
display(pd.DataFrame(peek.get('metadatas', [])))
assert collection.count() == len(chunks)

## 11. Vector Retrieval

`retrieve(query, top_k)` создаёт embedding запроса той же моделью и ищет top-k chunks в ChromaDB.

In [ ]:
def retrieve(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    query_embedding = embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    result = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
        include=['documents', 'metadatas', 'distances'],
    )
    rows: list[dict[str, Any]] = []
    for rank, (document, metadata, distance) in enumerate(zip(result['documents'][0], result['metadatas'][0], result['distances'][0]), start=1):
        rows.append({
            'rank': rank,
            'distance': float(distance),
            'source': metadata.get('source'),
            'breed_name': metadata.get('breed_name'),
            'breed_id': metadata.get('breed_id'),
            'chunk_id': metadata.get('chunk_id'),
            'doc_id': metadata.get('doc_id'),
            'wikipedia_url': metadata.get('wikipedia_url'),
            'text': document,
            'metadata': metadata,
        })
    return rows

def show_retrieval_results(rows: list[dict[str, Any]]) -> None:
    display(pd.DataFrame([
        {
            'rank': row['rank'],
            'distance': round(row['distance'], 4),
            'source': row['source'],
            'breed_name': row['breed_name'],
            'chunk_id': row['chunk_id'],
            'text_preview': row['text'][:260],
            'wikipedia_url': row['wikipedia_url'],
        }
        for row in rows
    ]))

for query in ['Какая порода кошек подходит для семьи?', 'Как ухаживать за сфинксом?', 'Large gentle cat with long coat']:
    print('\nQUERY:', query)
    retrieved = retrieve(query, top_k=TOP_K)
    show_retrieval_results(retrieved)

## 12. RAG Prompt

Собираем prompt с системной инструкцией, вопросом пользователя, retrieved context и ссылками на chunks.

In [ ]:
SYSTEM_INSTRUCTION = '\n'.join([
    'Ты дружелюбный ассистент по породам кошек.',
    'Отвечай на русском языке понятно для обычного пользователя.',
    'Используй только факты из retrieved CatAPI context.',
    'Не выдумывай медицинские рекомендации и диагнозы.',
    'Если данных недостаточно, честно скажи, что в контексте не хватает информации.',
    'В конце кратко перечисли использованные источники/chunk_id.',
])

def build_rag_prompt(query: str, retrieved_chunks: list[dict[str, Any]]) -> str:
    context_parts = []
    for row in retrieved_chunks:
        context_parts.append('\n'.join([
            f"Chunk ID: {row['chunk_id']}",
            f"Breed: {row['breed_name']}",
            f"Source: {row['source']}",
            f"Wikipedia: {row.get('wikipedia_url') or 'not provided'}",
            f"Text: {row['text']}",
        ]))
    context = '\n\n---\n\n'.join(context_parts)
    return (
        f"SYSTEM INSTRUCTION:\n{SYSTEM_INSTRUCTION}\n\n"
        f"USER QUESTION:\n{query}\n\n"
        f"RETRIEVED CATAPI CONTEXT:\n{context}\n\n"
        "ANSWER REQUIREMENTS:\n"
        "- Answer only using retrieved context.\n"
        "- If context is insufficient, say so.\n"
        "- Mention chunk IDs used as sources."
    )

sample_query = 'Чем британские котики отличаются от обычных?'
sample_retrieved = retrieve(sample_query, top_k=TOP_K)
sample_prompt = build_rag_prompt(sample_query, sample_retrieved)
print(sample_prompt)

## 13. Real Mistral API Call

Эта ячейка делает реальный запрос к Mistral. API key берётся из `MISTRAL_API_KEY` в environment/Kaggle Secrets. Если ключа нет, ячейка останавливается с понятной ошибкой. Mock answer не используется.

In [ ]:
if not MISTRAL_API_KEY:
    raise RuntimeError('MISTRAL_API_KEY is not configured. Add it to Kaggle Secrets or environment variables.')

mistral_client = Mistral(api_key=MISTRAL_API_KEY)
response = mistral_client.chat.complete(
    model=MISTRAL_MODEL,
    messages=[
        {'role': 'system', 'content': SYSTEM_INSTRUCTION},
        {'role': 'user', 'content': sample_prompt},
    ],
)

sample_answer = response.choices[0].message.content
if not sample_answer:
    raise RuntimeError('Mistral returned an empty answer.')

print('Question:', sample_query)
print('\nRetrieved chunks:')
show_retrieval_results(sample_retrieved)
print('\nMistral answer:\n')
print(sample_answer)

## 14. End-to-End Function

Функция выполняет полный RAG-путь: query → embedding → Chroma retrieval → context → Mistral request → structured result.

In [ ]:
def answer_with_rag(query: str, top_k: int = 5) -> dict[str, Any]:
    retrieved_chunks = retrieve(query, top_k=top_k)
    prompt = build_rag_prompt(query, retrieved_chunks)
    response = mistral_client.chat.complete(
        model=MISTRAL_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_INSTRUCTION},
            {'role': 'user', 'content': prompt},
        ],
    )
    answer = response.choices[0].message.content
    if not answer:
        raise RuntimeError('Mistral returned an empty answer.')
    sources = [
        {
            'chunk_id': row['chunk_id'],
            'breed_name': row['breed_name'],
            'source': row['source'],
            'wikipedia_url': row['wikipedia_url'],
            'distance': row['distance'],
        }
        for row in retrieved_chunks
    ]
    return {'query': query, 'answer': answer, 'sources': sources, 'retrieved_chunks': retrieved_chunks}

## 15. Final Demonstration

Три содержательных вопроса. Каждый делает настоящий vector retrieval и реальный вызов Mistral.

In [ ]:
for question in [
    'Какая порода кошек подходит для спокойного человека?',
    'Как ухаживать за сфинксом?',
    'Чем мейн-кун отличается от британской короткошёрстной?',
]:
    result = answer_with_rag(question, top_k=TOP_K)
    print('\n' + '=' * 100)
    print('QUESTION:', result['query'])
    print('\nRETRIEVED CHUNKS:')
    show_retrieval_results(result['retrieved_chunks'])
    print('\nANSWER:\n')
    print(result['answer'])
    print('\nSOURCES:')
    display(pd.DataFrame(result['sources']))

## 16. Smoke Test

Финальная автоматическая проверка ключевых объектов RAG-пайплайна.

In [ ]:
assert len(documents) > 0
assert len(chunks) > 0
assert len(embeddings) == len(chunks)
assert embeddings.ndim == 2
assert not np.isnan(embeddings).any()
assert collection.count() == len(chunks)

test_results = retrieve('Какая порода кошек подходит для семьи?', top_k=3)
assert len(test_results) > 0
assert len(test_results) <= 3

print('RAG smoke test passed.')